# 07. Content Builder Agent — AGENTS.md + Skills + Subagents

## Learning goals

- Understand the official Deep Agents Content Builder pattern: **memory, skills, and subagents**
- Inject brand voice with `AGENTS.md` and load writing workflows on demand with `SKILL.md`
- Persist outputs safely with `FilesystemBackend(virtual_mode=True)`
- Keep search and image generation optional so the core text pipeline runs with OpenAI only

## Overview

| Item | Details |
|------|---------|
| **Official pattern** | Content Builder Agent — memory(`AGENTS.md`), skills, subagents |
| **Runtime scope** | Text-only LinkedIn/blog draft generation |
| **Safety** | `local/` outputs, `virtual_mode=True`, image generation as reference-only |
| **Verification** | One live gpt-4.1 harness run |

The official example includes Tavily search and Gemini image generation. This notebook keeps those as optional adapters and makes **OpenAI + local filesystem** the only required runtime path.

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY in .env"

In [ ]:
# LangSmith — automatic logging when LANGSMITH_TRACING=true
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGSMITH_PROJECT", "langchain-langgraph-deepagents-notebooks")

In [ ]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-5.4", temperature=0)

## 1) Content Builder workspace

The official example keeps agent instructions on disk rather than hard-coding everything in Python.

- `AGENTS.md` — brand voice and writing standards, loaded every run
- `skills/content-builder/SKILL.md` — writing workflow, loaded on demand
- `research/`, `linkedin/`, `blogs/` — output folders created by the agent

In [ ]:
from pathlib import Path
import shutil

WORK_DIR = Path("local/content_builder_demo_en")
if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
(WORK_DIR / "skills/content-builder").mkdir(parents=True)
(WORK_DIR / "research").mkdir()

In [ ]:
brand_voice = """# BAEUM.AI Content Voice
Write in English, professional but approachable.
Lead with practical value, keep paragraphs short, and end with action.
Focus on AI agents, developer productivity, and production reliability.
"""
(WORK_DIR / "AGENTS.md").write_text(brand_voice, encoding="utf-8")

In [ ]:
skill_text = """---
name: content-builder
description: Use for blog, LinkedIn, or technical content drafts.
---
# Content Builder
Research first, then write a hook, 3 concise insights, and a CTA.
Save LinkedIn posts to linkedin/<slug>/post.md.
Save blog posts to blogs/<slug>/post.md.
"""
(WORK_DIR / "skills/content-builder/SKILL.md").write_text(skill_text, encoding="utf-8")

## 2) Local research tool and researcher subagent

Live classrooms and automated smoke tests do not always have external search keys. Here we wrap a small local knowledge base as a tool, then let a researcher subagent save concise findings to disk.

In [ ]:
from langchain.tools import tool

@tool
def topic_brief(topic: str) -> str:
    """Return a concise local research brief for a content topic."""
    return (
        f"{topic}: Deep Agents combine memory, skills, filesystem tools, "
        "and subagents so content workflows become repeatable."
    )

In [ ]:
researcher = {
    "name": "researcher",
    "description": "Research a topic and save concise findings before writing.",
    "system_prompt": "Use topic_brief, then write findings to the requested path.",
    "tools": [topic_brief],
}

## 3) Configure the Deep Agent

`create_deep_agent()` wires the content-building pieces together.

| Configuration | Role |
|------|------|
| `memory=["/AGENTS.md"]` | Injects brand voice into the system prompt |
| `skills=["/skills/"]` | Loads `SKILL.md` workflow guidance on demand |
| `subagents=[researcher]` | Delegates research through the `task` tool |
| `FilesystemBackend` | Provides file tools such as `write_file` and `read_file` |

In [ ]:
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend

agent = create_deep_agent(
    model=model, tools=[topic_brief], subagents=[researcher],
    backend=FilesystemBackend(root_dir=str(WORK_DIR), virtual_mode=True),
    memory=["/AGENTS.md"], skills=["/skills/"],
    system_prompt="You are a content builder. Use files for durable outputs.",
)

## 4) Run — create a LinkedIn post

The request states four things explicitly: use the researcher, where to save research, where to save the content, and the bounded output shape.

In [ ]:
request = """Use the researcher subagent to research Deep Agents content workflows.
Save research to research/content-workflows.md.
Then write an English LinkedIn post with one hook, three bullets, and one CTA.
Save it to linkedin/content-workflows/post.md."""
result = agent.invoke({"messages": [{"role": "user", "content": request}]})
print(result["messages"][-1].content[:800])

In [ ]:
post_path = WORK_DIR / "linkedin/content-workflows/post.md"
research_path = WORK_DIR / "research/content-workflows.md"
print("research exists:", research_path.exists())
print("post exists:", post_path.exists())
print(post_path.read_text(encoding="utf-8")[:900])

## 5) Search and images are optional adapters

A production Content Builder can add Tavily search and image generation tools. For the core 01~07 notebook harness, keep those adapters reference-only unless the matching keys and cost policy are available.

In [ ]:
optional_tools_example = r"""
# Optional production tools — not required for this notebook smoke run.
# Add Tavily or image generation only when the matching API keys exist.
agent = create_deep_agent(
    model=model,
    tools=[web_search, generate_cover, generate_social_image],
    memory=["/AGENTS.md"], skills=["/skills/"],
)
"""
print(optional_tools_example)

## Summary

| Topic | Key idea |
|------|----------|
| **Memory** | `AGENTS.md` always injects brand voice |
| **Skills** | `SKILL.md` loads blog/social workflows on demand |
| **Subagents** | A researcher separates evidence gathering from drafting |
| **FilesystemBackend** | Durable outputs are saved under `local/` |
| **Runtime policy** | OpenAI is required; search and images stay optional |

---

**References:**
- Deep Agents Content Builder: https://docs.langchain.com/oss/python/deepagents/content-builder
- Deep Agents Skills: ../../docs/deepagents/10-skills.md
- Deep Agents Subagents: ../../docs/deepagents/07-subagents.md
- Local reference: ../../docs/deepagents/examples/01-content-builder-agent.md